# Results Checking

Thorough audit of the result files: **coverage** (what the config says should exist vs. what's on disk), **completeness** (folds < `cv_splits`, malformed files), **sanity** (NaN / out-of-range / worse-than-random metrics), **stale** files, and **visualisations** (dataset×method coverage heatmaps, fold completeness, metric distributions, sweep curves).

All logic lives in [`src/utils/results_checking.py`](../src/utils/results_checking.py); this notebook is a thin viewer. It transparently handles **both** result layouts: one file per `(dataset, method)` point (Experiment 0/1), and **packed** files where all of a cell's sweep points live in a single `<method>.json` (Experiment 2/3, metrics only — no prediction arrays).

> **Where to run:** wherever the results live. On the cluster the results are on the Lustre **project storage** (`/staging/leuven/stg_00211/results`), which the OnDemand *file browser* cannot see — open a **VSCode session in OnDemand** (it *can* see Lustre) and run this notebook there, or run it locally against a downloaded copy. Set `RESULTS_ROOT` below accordingly.

In [ ]:
import sys
from pathlib import Path

# Make `import src.*` work when running from the notebooks/ folder.
_REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from src.utils import results_checking as rc
from src.utils.paths import results_root

# None -> auto-resolve ($TABPFN_RESULTS_ROOT / project-storage / ./results).
# On the cluster you can pin it explicitly, e.g.:
#   RESULTS_ROOT = '/staging/leuven/stg_00211/results'
RESULTS_ROOT = None
print('Results root:', RESULTS_ROOT or results_root())

## 1. One-call overview (all experiments)

Headline tables + plots for every experiment. Drill into any one in the sections below.

In [ ]:
audits = rc.run_full_audit(
    experiments=('Experiment0', 'Experiment1', 'Experiment2', 'Experiment3'),
    results_root=RESULTS_ROOT,
    show_plots=True,
)

## 2. Drill into one experiment

Set `EXP` and re-run the cells below.

In [ ]:
EXP = 'Experiment2'
audit = rc.audit_experiment(EXP, results_root=RESULTS_ROOT)
audit.summary()

In [ ]:
# What's MISSING (expected by the config but absent on disk)
audit.missing_frame()

In [ ]:
# INCOMPLETE (fewer folds than cv_splits) and MALFORMED / unreadable files
display(audit.incomplete_frame())
from src.utils.results_checking import _combo_frame
display(_combo_frame(audit.malformed))

In [ ]:
# ANOMALOUS results (NaN/inf, out-of-range, worse-than-random AUC, ...)
audit.anomalies_frame()

In [ ]:
# STALE files on disk that are NOT in the current config (disabled method, old sweep, ...)
audit.stale_frame()

## 3. Visualise coverage

In [ ]:
# dataset x method completion fraction (1.0 green = done, 0.0 red = missing)
for task in ('pd', 'lgd'):
    rc.plot_coverage_heatmap(audit, task)

In [ ]:
rc.plot_fold_completeness(audit)

## 4. Sanity-check the metric values

In [ ]:
rc.plot_metric_distributions(audit, metric='AUC')   # PD; use 'R2' for LGD

In [ ]:
# Exp 2 (learning curve) / Exp 3 (imbalance): metric vs sweep value, per (dataset, method).
# Gaps = missing sweep points; non-monotonic jumps = worth a look.
rc.plot_sweep_curves(audit, task='pd', metric='AUC')

## 5. Raw table for custom slicing

One row per result file (with its mean metrics) — slice / group / export as you like.

In [ ]:
df = audit.table
print(df.shape)
df.head(50)

In [ ]:
# e.g. export the full audit table
# df.to_csv(f'{EXP.lower()}_audit.csv', index=False)